In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ===============================
# Load Dataset
# ===============================
df = pd.read_csv("yield_df.csv")

# ===============================
# Drop Unnecessary Column
# ===============================
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

# ===============================
# Label Encoding
# ===============================
area_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df["Area"] = area_encoder.fit_transform(df["Area"])
df["Item"] = item_encoder.fit_transform(df["Item"])

# ===============================
# Features and Target
# ===============================
X = df.drop("hg/ha_yield", axis=1)
y = df["hg/ha_yield"]

# ===============================
# Train Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ===============================
# Random Forest Regressor
# ===============================
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# ===============================
# Prediction
# ===============================
y_pred = rf.predict(X_test)

# ===============================
# Evaluation
# ===============================
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("=" * 50)
print("Random Forest Regressor Results")
print("=" * 50)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

# ===============================
# Feature Importance
# ===============================
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance\n")
print(importance)

# ===============================
# Save Model
# ===============================
joblib.dump(rf, "random_forest_regressor.pkl")
joblib.dump(area_encoder, "area_encoder.pkl")
joblib.dump(item_encoder, "item_encoder.pkl")

print("\nModel Saved Successfully!")

Random Forest Regressor Results
MAE  : 3752.53
RMSE : 10118.05
R²   : 0.9859

Feature Importance

                         Feature  Importance
1                           Item    0.609017
4              pesticides_tonnes    0.110008
5                       avg_temp    0.109030
3  average_rain_fall_mm_per_year    0.086008
0                           Area    0.055120
2                           Year    0.030817

Model Saved Successfully!
